Christina Ndayisenga_MSBD566_09/22/2026 I confirm that this assignment reflects my own work and understanding.

A: The business problem for this dataset is identifying patients who are at a higher risk of being readmitted to the hospital within 30 days after discharge. By identifying these patients early, healthcare providers can take steps to reduce the chances of readmission. This could include providing additional discharge planning, follow-up appointments, or other interventions based on the patient's needs. The overall goal is to improve patient outcomes while reducing unnecessary hospital readmissions and healthcare costs. I would adapt this problem by focusing on patients aged 65 and older to identify those at a higher risk of being readmitted within 30 days. This could help healthcare providers develop more targeted interventions for older adults who may need additional care and support after discharge.

B: For the train/test split, we chose to group the data by patient so that the same patient would not appear in both the training and testing datasets. An alternative approach would have been to randomly split individual hospital encounters, which could allow the same patient to appear in both groups. We chose the patient-grouped approach because it helps prevent data leakage and gives us a more realistic evaluation of how the model performs on new patients. However, this decision could affect model performance if patients with frequent hospital visits are not equally represented in both groups. For example, if the training set contains fewer patients with multiple diabetes-related admissions, the model can have difficulty predicting readmission for patients with a history of frequent hospitalizations.

For the insulin variable, we chose to treat it as a nominal variable instead of an ordinal variable because the categories represent different changes in insulin treatment rather than a clear order. An alternative approach would have been to assign numerical values to categories such as No, stable, up, and down. However, this could make the model think that the categories have a specific ranking or have equal distance between them. One way this decision could affect model performance is if changes in insulin treatment are related to a patient's risk of readmission. Like patients whose insulin dosage increased or decreased during hospitalization could have different readmission risks, and treating the categories separately could make it harder for some models to recognize patterns related to the direction of treatment changes.

C: I will compare logistic regression and random forest to predict 30-day hospital readmission. Logistic regression will help me understand how different patient characteristics are related to readmission risk, while random forest can identify more complex patterns in the data. Comparing both models will help me determine how well each one predicts readmission.

I will use precision, recall, and F1-score to evaluate the models. Since the dataset has more patients who are not readmitted than patients who are readmitted, accuracy alone could be misleading. Recall will be especially important because I want to identify as many high-risk patients as possible, while precision will help me understand how many patients identified as high risk are actually readmitted.

One risk I will watch for is data leakage, especially because some patients have multiple hospital encounters in the dataset. To prevent this, I will use the patient-grouped train/test split so that the same patient does not appear in both groups. This will help me get a more realistic understanding of how well the models perform when predicting readmission for new patients.

In [44]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np

In [35]:
X_train = pd.read_csv("/Users/christinandayisenga/Downloads/X_train.csv")
X_test = pd.read_csv("/Users/christinandayisenga/Downloads/X_test.csv")

y_train = pd.read_csv("/Users/christinandayisenga/Downloads/y_train.csv").squeeze("columns")
y_test = pd.read_csv("/Users/christinandayisenga/Downloads/y_test.csv").squeeze("columns")

In [36]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("Predictions:", len(lr_predictions))

X_train: (79567, 42)
X_test: (19773, 42)
y_train: (79567,)
y_test: (19773,)
Predictions: 19773


In [40]:
lr_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

# Train the model

lr_model.fit(X_train, y_train)

# Make predictions on the test data

lr_predictions = lr_model.predict(X_test)

# Print the predictions counts

print("Logistic Regression predictions:")
print(pd.Series(lr_predictions).value_counts())

Logistic Regression predictions:
0    19738
1       35
Name: count, dtype: int64


In [41]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# train the model

rf_model.fit(X_train, y_train)

# make predictions on the test data

rf_predictions = rf_model.predict(X_test)

# Print the predictions counts

print("Random forest predictions:")
print(pd.Series(rf_predictions).value_counts())

Random forest predictions:
0    19719
1       54
Name: count, dtype: int64


In [39]:
print(confusion_matrix(y_test, lr_predictions))
print(classification_report(y_test, lr_predictions, zero_division=0))

[[17520     9]
 [ 2218    26]]
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     17529
           1       0.74      0.01      0.02      2244

    accuracy                           0.89     19773
   macro avg       0.82      0.51      0.48     19773
weighted avg       0.87      0.89      0.84     19773



In [49]:
# Compare both models on the same test set
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Precision": [
        precision_score(y_test, lr_predictions, zero_division=0),
        precision_score(y_test, rf_predictions, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, lr_predictions, zero_division=0),
        recall_score(y_test, rf_predictions, zero_division=0)
    ],
    "F1-score": [
        f1_score(y_test, lr_predictions, zero_division=0),
        f1_score(y_test, rf_predictions, zero_division=0)
    ],
    "Accuracy": [
        accuracy_score(y_test, lr_predictions),
        accuracy_score(y_test, rf_predictions)
    ]
})

print(results.round(3).to_string(index=False))

              Model  Precision  Recall  F1-score  Accuracy
Logistic Regression      0.743   0.012     0.023     0.887
      Random Forest      0.537   0.013     0.025     0.887


E: Both models had 88.7% accuracy. Logistic Regression had higher precision (74.3%), while Random Forest had slightly higher recall (1.3%) and F1-score (0.025).
I would choose Logistic Regression to keep working with because its precision was higher. However, both models missed almost all the patients who were readmitted, so I would need to improve recall before using either model.
I was surprised that both models had high accuracy but very low recall. I also expected Random Forest to perform better, but its results were very similar to Logistic Regression’s.

F: With more time, I would try different model settings and ways to handle the class imbalance to improve recall. I would also use more patient data to see if the models could identify patterns related to readmission. I was surprised that both models had high accuracy but missed almost all the patients who were actually readmitted. This showed me why it is important to look at more than just accuracy when evaluating a model.